# Lesson 06: Production-Shaped Measurements

The initial target in this project is provider-call p95 under 200 ms. That is not the same as end-to-end guardrail latency. A production integration adds preprocessing, routing, policy logic, and application/network overhead. Start in shadow mode and compare decisions before enforcing them.

## Measure distinct stages

Record provider-call p50/p95/p99 and separately measure total guardrail time from receiving the input through making the policy decision. Run from a documented region, at expected concurrency, with a warm-up and enough requests to characterize tail behavior. A tiny pilot cannot prove an SLO.

In [ ]:
from jev_eval.metrics import latency_metrics

# Synthetic measurements to demonstrate reporting only.
provider_ms = [92, 111, 130, 148, 176, 205, 241, 390]
end_to_end_ms = [provider + 18 for provider in provider_ms]
print("SYNTHETIC values, not benchmark results")
print({"provider": latency_metrics(provider_ms), "end_to_end": latency_metrics(end_to_end_ms)})

## Decide what happens on uncertainty and outage

A guardrail policy should explicitly map scores to allow/review/block, and API errors to a separate operational outcome. For high-impact actions, fail closed or require review; for availability-sensitive paths, define a bounded fallback and log the bypass. Never make an outage look like a low-risk model score. Keep tool permissions and user confirmation as independent controls.

In [ ]:
def disposition(score: float, *, allow_below: float, block_at: float) -> str:
    if allow_below >= block_at:
        raise ValueError("allow_below must be less than block_at")
    if score < allow_below:
        return "allow_candidate"
    if score >= block_at:
        return "block_candidate"
    return "review"

[(score, disposition(score, allow_below=0.2, block_at=0.7)) for score in (0.05, 0.45, 0.91)]